# Sesión 3 · Datos tabulares I: leer un CSV difícil y una API

**Programación avanzada en Python aplicada al análisis social** · miércoles 9 de diciembre de 2026

Objetivos:

1. Leer un CSV «tal como viene» de un organismo oficial y dejarlo limpio.
2. Entender qué es una **API** y descargar datos de una, paso a paso.
3. Guardar una copia local y escribir una función que no falle si no hay conexión.

Al terminar tendréis **vuestras propias funciones** para leer y descargar la serie de hipotecas del INE.

In [ ]:
# Preparacion: localiza la carpeta de la asignatura (y la descarga si estas en Google Colab).
# Ejecuta esta celda primero; no hace falta entenderla todavia.
import os, subprocess
from pathlib import Path

REPO = "https://github.com/luisaugustos/ppy-analisis-social.git"


def _buscar_carpeta():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        for candidata in (carpeta, carpeta / "ppy-analisis-social"):
            if (candidata / "datos" / "ORIGEN.md").is_file():
                return candidata
    return None


raiz = _buscar_carpeta()
if raiz is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    raiz = Path("ppy-analisis-social").resolve()

os.chdir(raiz)
print("Carpeta de trabajo lista. Los datos estan en la subcarpeta 'datos'.")

## 1. Un CSV tal como viene

Hasta ahora hemos leído ficheros limpios. Los que se descargan de la web de un organismo no lo están tanto. Lo primero es **mirar el fichero como texto**, sin pandas:

In [ ]:
from pathlib import Path

ruta = Path("datos/ine_hipotecas_bruto.csv")

with open(ruta, encoding="latin-1") as fichero:
    for numero in range(5):
        print(fichero.readline().strip())

Cinco cosas que hay que decirle a `read_csv`:

| Lo que vemos | Argumento |
| --- | --- |
| Los valores se separan con punto y coma | `sep=";"` |
| Los miles llevan punto: `44.300` | `thousands="."` |
| Los valores que faltan se escriben `..` | `na_values=".."` |
| El fichero no está en UTF-8 (tildes) | `encoding="latin-1"` |
| Los decimales, cuando los hay, llevan coma | `decimal=","` |

In [ ]:
import pandas as pd

bruto = pd.read_csv(ruta, sep=";", thousands=".", decimal=",", na_values="..", encoding="latin-1")
bruto.head()

In [ ]:
bruto.dtypes

> **Si las tildes salen mal**
>
>  Si en lugar de las tildes aparecen símbolos extraños, la codificación no es la correcta: probad `encoding="latin-1"` o `encoding="utf-8"`.

## 2. Dejarlo limpio, paso a paso

Queremos una tabla con una fila por provincia y mes, y columnas `cod_prov`, `provincia`, `fecha`, `hipotecas` e `importe_miles`. Vamos de uno en uno.

**Paso 1: separar el código y el nombre de la provincia.** Los métodos de texto de pandas están detrás de `.str`.

In [ ]:
bruto["Provincias"].head(3)

In [ ]:
es_nacional = bruto["Provincias"] == "Total Nacional"

bruto["cod_prov"] = bruto["Provincias"].str[:2]            # los dos primeros caracteres
bruto["provincia"] = bruto["Provincias"].str[3:]           # del cuarto en adelante
bruto.loc[es_nacional, "cod_prov"] = "00"                  # el total nacional no trae código
bruto.loc[es_nacional, "provincia"] = "Total Nacional"

bruto[["Provincias", "cod_prov", "provincia"]].drop_duplicates().head(4)

**Paso 2: convertir el periodo en una fecha.** `2026M08` significa agosto de 2026.

In [ ]:
bruto["fecha"] = pd.to_datetime(bruto["Periodo"], format="%YM%m")
bruto[["Periodo", "fecha"]].head(3)

**Paso 3: una columna por concepto.** El número y el importe vienen en filas distintas; `pivot` los pone en columnas.

In [ ]:
limpio = bruto.pivot(index=["cod_prov", "provincia", "fecha"], columns="Concepto", values="Total").reset_index()
limpio = limpio.rename(columns={"Número de hipotecas": "hipotecas", "Importe de hipotecas": "importe_miles"})
limpio.columns.name = None
limpio.head()

### Todo junto, en una función

Los pasos funcionan: es el momento de darles nombre, como hicimos en la sesión 2.

In [ ]:
def leer_hipotecas_ine(ruta):
    """Lee el CSV de hipotecas descargado de la web del INE y lo devuelve limpio.

    Resultado: una fila por provincia y mes, con las columnas
    cod_prov, provincia, fecha, hipotecas e importe_miles.
    """
    bruto = pd.read_csv(ruta, sep=";", thousands=".", decimal=",", na_values="..", encoding="latin-1")

    es_nacional = bruto["Provincias"] == "Total Nacional"
    bruto["cod_prov"] = bruto["Provincias"].str[:2]
    bruto["provincia"] = bruto["Provincias"].str[3:]
    bruto.loc[es_nacional, "cod_prov"] = "00"
    bruto.loc[es_nacional, "provincia"] = "Total Nacional"
    bruto["fecha"] = pd.to_datetime(bruto["Periodo"], format="%YM%m")

    limpio = bruto.pivot(index=["cod_prov", "provincia", "fecha"], columns="Concepto", values="Total").reset_index()
    limpio = limpio.rename(columns={"Número de hipotecas": "hipotecas", "Importe de hipotecas": "importe_miles"})
    limpio.columns.name = None
    return limpio


hipotecas = leer_hipotecas_ine("datos/ine_hipotecas_bruto.csv")
hipotecas.tail(3)

## 3. Trabajar con fechas

Cuando la columna de fechas se usa como **índice**, pandas entiende de meses y años.

In [ ]:
salamanca = hipotecas[hipotecas["provincia"] == "Salamanca"].set_index("fecha")["hipotecas"]

print(salamanca.loc["2025-03"])                  # un mes
print(salamanca.loc["2025"].sum())               # todo un año
salamanca.resample("YS").sum()                   # de mensual a anual

> **Cuidado con el último año**
>
>  El último año suele estar incompleto. Antes de comparar totales anuales, comprobad cuántos meses tiene cada año con `.resample("YS").count()`.

## 4. ¿Qué es una API?

Descargar el CSV a mano cada mes es pesado y se olvida. Muchos organismos ofrecen una **API**: una dirección web pensada para que la consulte un programa en lugar de una persona. Le pedimos algo con una dirección y nos responde con datos.

La del INE tiene esta forma:

```
https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/3200?tip=AM&nult=3
```

| Parte | Significado |
| --- | --- |
| `servicios.ine.es/wstempus/js/ES` | La dirección de la API, en español |
| `DATOS_TABLA/3200` | Qué pedimos: los datos de la tabla 3200 (hipotecas por provincia) |
| `tip=AM` | Con nombres legibles |
| `nult=3` | Solo los tres últimos periodos |

Podéis pegar esa dirección en el navegador para ver la respuesta.

### Pedir los datos con `requests`

In [ ]:
import json

import requests

url = "https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/3200?tip=AM&nult=3"

try:
    respuesta = requests.get(url, timeout=30)
    respuesta.raise_for_status()                     # da error si el servidor responde mal
    series = respuesta.json()                        # convierte la respuesta en listas y diccionarios
    print("Respuesta recibida del INE")
except requests.RequestException as error:
    print("No se ha podido consultar la API:", type(error).__name__)
    series = json.loads(Path("datos/ine_respuesta_ejemplo.json").read_text(encoding="utf-8"))
    print("Uso la respuesta de ejemplo guardada en la carpeta datos")

### Explorar la respuesta

La respuesta llega en formato **JSON**, que en Python se convierte en lo que ya conocéis: listas y diccionarios. Para entenderla, se va abriendo poco a poco.

In [ ]:
print(type(series), len(series))        # una lista de series

In [ ]:
primera = series[0]                     # cada serie es un diccionario
print(primera.keys())

In [ ]:
print(primera["Nombre"])
print(primera["Data"][0])               # 'Data' es una lista con un diccionario por periodo

Cada **serie** es una combinación de territorio y concepto (por ejemplo, «número de hipotecas en Salamanca»), y dentro de `Data` hay un valor por mes.

### Del JSON a una tabla

Recorremos las series y, dentro de cada una, sus datos. Con cada valor formamos una fila (un diccionario) y las juntamos en una lista.

In [ ]:
filas = []
for serie in series:
    for dato in serie["Data"]:
        filas.append({
            "nombre": serie["Nombre"],
            "fecha": dato["Fecha"][:10],          # nos quedamos con 'AAAA-MM-DD'
            "valor": dato["Valor"],
        })

tabla = pd.DataFrame(filas)
tabla["fecha"] = pd.to_datetime(tabla["fecha"])
tabla.head()

Una lista de diccionarios se convierte en tabla con `pd.DataFrame(...)`. Es el patrón general para cualquier API.

## 5. Una función de descarga que no falla

Juntamos todo en una función con tres ideas:

1. Si ya tenemos una **copia local**, la leemos y no molestamos al servidor.
2. Si no, **descargamos** y guardamos la copia.
3. Si la descarga falla, lo decimos con un **mensaje claro**.

In [ ]:
def descargar_tabla_ine(id_tabla, ultimos=None, carpeta="datos/descargas"):
    """Descarga una tabla de la API del INE y la devuelve como tabla de pandas.

    Guarda una copia en 'carpeta'; si la copia ya existe, la lee en lugar de descargar.
    'ultimos' limita la descarga a los últimos periodos.
    """
    copia = Path(carpeta) / f"ine_{id_tabla}.csv"
    if copia.exists():
        return pd.read_csv(copia, parse_dates=["fecha"])

    url = f"https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/{id_tabla}?tip=AM"
    if ultimos:
        url += f"&nult={ultimos}"
    respuesta = requests.get(url, timeout=60)
    respuesta.raise_for_status()

    filas = []
    for serie in respuesta.json():
        for dato in serie["Data"]:
            filas.append({"nombre": serie["Nombre"], "fecha": dato["Fecha"][:10], "valor": dato["Valor"]})
    tabla = pd.DataFrame(filas)
    tabla["fecha"] = pd.to_datetime(tabla["fecha"])

    copia.parent.mkdir(parents=True, exist_ok=True)
    tabla.to_csv(copia, index=False)
    return tabla

In [ ]:
try:
    recientes = descargar_tabla_ine(3200, ultimos=3)
    print(len(recientes), "filas")
except requests.RequestException as error:
    print("Sin conexión con el INE:", type(error).__name__)

> **Buenas maneras con una API**
>
>  - **Guardad copia** y no descarguéis lo mismo una y otra vez.
> - Leed las condiciones de uso y **citad la fuente**.
> - Poned siempre un `timeout`, para que el programa no se quede esperando indefinidamente.

## Resumen

- Antes de leer un CSV, **miradlo como texto** y traducid lo que veis a argumentos de `read_csv`.
- Limpiad **paso a paso**, comprobando cada uno, y después juntad los pasos en una función.
- Una API es una dirección que responde con datos; `requests.get(url).json()` devuelve listas y diccionarios.
- Lista de diccionarios → `pd.DataFrame`.
- Copia local primero; descarga solo si hace falta.

En *Obtención, almacenamiento y preprocesamiento* veis otras formas de conseguir datos (bases de datos, *scraping*). Las funciones de hoy se quedan con lo más habitual en el día a día: un CSV y una API pública.